# The judge vs the offline ASR, side by side

**What this shows.** The same audio, read by two listeners: `faster-whisper small.en`
(the offline ASR standing in for the judge until now) and the live judge
(`gemini-3.7-flash`, audio in / text out).

**Why it matters.** `metric-definitions.md` §1 claims conventional TSE can improve
offline transcription while making audio *harder for a live model*. Both halves of
that sentence were the same ASR until the judge existed, so the claim was
untestable. This notebook is where the two listeners can finally disagree.

**Read the error split, not just the total.** Word error rate is
`(substitutions + deletions + insertions) / reference words`. Two listeners can
reach the same total by completely different routes -- one omitting words, the
other inventing them -- and that difference is the finding.

Scoring uses the project's own `lcf_wer.count_errors`, so these numbers are the
metric's numbers, not a reimplementation.

In [8]:
import csv, json, sys
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO))

from src.live_model_metric.lcf_wer import count_errors, normalise_text

SPLIT = "sir0_val"
JUDGE_CACHE = REPO / "experiments/results/judge_responses.csv"
ASR_CACHE   = REPO / "experiments/results/transcripts.csv"

judge_rows = list(csv.DictReader(open(JUDGE_CACHE)))
asr = {(r["trial_id"], r["file"]): r["text"]
       for r in csv.DictReader(open(ASR_CACHE))}
manifest = {r["trial_id"]: r
            for r in csv.DictReader(open(REPO / f"data/manifests/{SPLIT}.csv"))}

print(f"judge responses cached : {len(judge_rows)}")
print(f"models seen            : {sorted({r['model'] for r in judge_rows})}")
print(f"prompt hashes seen     : {sorted({r['prompt_sha'] for r in judge_rows})}")
print(f"run dates              : {sorted({r['run_date'] for r in judge_rows})}")

judge responses cached : 24
models seen            : ['gemini-3.7-flash']
prompt hashes seen     : ['d118b7d3bf30']
run dates              : ['2026-09-02']


**One prompt hash and one model, or the rows are not comparable.** The prompt and
the model ID are components of the measuring instrument. If more than one of either
appears above, the cache spans two different instruments and must be filtered before
anything is aggregated.

In [9]:
SYSTEM_OF = {"mixture.wav": "floor", "target.wav": "ceiling", "estimate.wav": "estimate"}

records = []
for r in judge_rows:
    tid, clip = r["trial_id"], r["file"]
    meta_path = REPO / f"data/rendered/{SPLIT}/{tid}/meta.json"
    if not meta_path.exists():
        continue
    meta = json.loads(meta_path.read_text())
    row = manifest.get(tid, {})
    absent = row.get("target_absent") == "1"

    reference = meta.get("target_text", "")
    system = SYSTEM_OF.get(clip, clip)
    # An absent trial's clean target is digital silence (measured RMS 0), so
    # the only correct answer is no words at all.
    if absent and clip == "target.wav":
        system = "silence"
    elif absent:
        system = "floor(absent)"

    records.append({
        "trial_id": tid, "clip": clip, "system": system,
        "condition": row.get("condition", "?"),
        "absent": absent,
        "sir_db": float(row["sir_db"]) if row.get("sir_db") else None,
        "reference": reference,
        "interferer": meta.get("interferer_text", ""),
        "judge_status": r["status"],
        "judge": r["text"],
        "asr": asr.get((tid, clip)),
    })

print(f"{len(records)} clips with a judge response")
for s in sorted({x['system'] for x in records}):
    print(f"  {s:<16} {sum(1 for x in records if x['system'] == s)}")

24 clips with a judge response
  ceiling          10
  floor            10
  floor(absent)    2
  silence          2


## 1. The side-by-side

Reference, then what each listener heard. This is the part worth reading with your
eyes rather than summarising -- the *character* of the mistakes is the point.

In [10]:
def wrap(text, width=96, indent=" " * 14):
    text = (text or "").strip()
    if not text:
        return indent + "(nothing)"
    words, lines, cur = text.split(), [], ""
    for w in words:
        if len(cur) + len(w) + 1 > width:
            lines.append(cur); cur = w
        else:
            cur = f"{cur} {w}".strip()
    lines.append(cur)
    return "\n".join(indent + l for l in lines)

def show(rec):
    head = f"{rec['trial_id']}  [{rec['system']}]"
    if rec["sir_db"] is not None:
        head += f"   SIR {rec['sir_db']:+.1f} dB"
    print("=" * 110)
    print(head)
    print("  REFERENCE :"); print(wrap(rec["reference"]))
    print("  small.en  :"); print(wrap(rec["asr"]))
    print(f"  JUDGE     : [{rec['judge_status']}]"); print(wrap(rec["judge"]))
    if rec["interferer"]:
        print("  (interferer said):"); print(wrap(rec["interferer"]))
    print()

for rec in sorted(records, key=lambda r: (r["system"], r["trial_id"])):
    show(rec)

sir0_val-42-000002  [ceiling]   SIR -1.2 dB
  REFERENCE :
              HARRY GAVE HIS FAREWELLS WITH DEEP AND GENUINE REGRET WHETHER THEIR MANNER WAS GRAVE OR
              FRIVOLOUS HE KNEW THAT THESE WERE GOOD FRIENDS OF HIS AND HE SINCERELY HOPED THAT HE WOULD MEET
              THEM AGAIN
  small.en  :
              Harry gave his farewells with deep and genuine regret. Whether their manner was grave or
              frivolous, he knew that these were good friends of his, and he sincerely hoped that he would
              meet them again.
  JUDGE     : [speech]
              Harry gave his farewells with deep and genuine regret. Whether their manner was grave or
              frivolous, he knew that these were good friends of his, and he sincerely hoped that he would
              meet them again.
  (interferer said):
              TO HIM THE PRESENCE OR ABSENCE OF HIS WIFE'S SISTER WAS A MATTER OF INDIFFERENCE HE WAS OF A
              CLEAN SAVING DISPOSITION AND HAD ALREADY PAI

## 2. Word error rate, and where the errors come from

`count_errors` returns substitutions, deletions and insertions separately. Absent
trials have no reference words, so their rate is 0/0 -- undefined, not perfect --
and they are **excluded** here rather than scored as zero (B4).

In [11]:
import pandas as pd

def score(reference, hypothesis):
    e = count_errors(reference, hypothesis)
    if e.reference_word_count == 0:
        return None
    return {"wer": e.total_errors / e.reference_word_count * 100.0,
            "sub": e.substitutions, "del": e.deletions, "ins": e.insertions,
            "n": e.reference_word_count}

table = []
for rec in records:
    if rec["absent"]:
        continue                     # no reference words -- see the silence section
    j, a = score(rec["reference"], rec["judge"]), score(rec["reference"], rec["asr"])
    if not j or not a:
        continue
    table.append({
        "trial": rec["trial_id"].replace("sir0_val-42-", ""),
        "system": rec["system"],
        "judge_wer": round(j["wer"], 1), "asr_wer": round(a["wer"], 1),
        "delta": round(j["wer"] - a["wer"], 1),
        "judge_S": j["sub"], "judge_D": j["del"], "judge_I": j["ins"],
        "asr_S": a["sub"], "asr_D": a["del"], "asr_I": a["ins"],
        "n": j["n"],
    })

frame = pd.DataFrame(table).sort_values(["system", "trial"]).reset_index(drop=True)
frame

,trial,system,judge_wer,asr_wer,delta,judge_S,judge_D,judge_I,asr_S,asr_D,asr_I,n
0,000002,ceiling,0.0,0.0,0.0,0,0,0,0,0,0,35
1,000004,ceiling,0.0,2.0,-2.0,0,0,0,1,0,0,50
2,000005,ceiling,0.0,2.4,-2.4,0,0,0,1,0,0,41
3,000008,ceiling,0.0,7.3,-7.3,0,0,0,3,0,0,41
4,000010,ceiling,0.0,6.2,-6.2,0,0,0,2,0,0,32
5,000011,ceiling,9.5,9.5,0.0,2,0,0,2,0,0,21
6,000012,ceiling,0.0,0.0,0.0,0,0,0,0,0,0,11
7,000014,ceiling,0.0,0.0,0.0,0,0,0,0,0,0,31
8,000015,ceiling,4.0,8.0,-4.0,1,0,0,1,0,1,25
9,000019,ceiling,2.3,2.3,0.0,1,0,0,1,0,0,44


**`delta` is judge minus ASR, in percentage points.** Negative means the judge
recovered more of what the target said; positive means it recovered less. Either
direction is a result -- what would be fatal is `delta` sitting at zero everywhere,
because that would mean the judge is just a costlier ASR and LCF-WER measures
nothing new.

In [12]:
for system, group in frame.groupby("system"):
    print(f"--- {system}  (n={len(group)}) ---")
    print(f"  judge  mean WER {group['judge_wer'].mean():6.1f} %")
    print(f"  small.en mean WER {group['asr_wer'].mean():6.1f} %")
    print(f"  mean delta       {group['delta'].mean():+6.1f} pts"
          f"   (range {group['delta'].min():+.1f} to {group['delta'].max():+.1f})")
    agree = (group["delta"].abs() <= 1.0).sum()
    print(f"  clips where they agree within 1 pt: {agree}/{len(group)}")
    print()

--- ceiling  (n=10) ---
  judge  mean WER    1.6 %
  small.en mean WER    3.8 %
  mean delta         -2.2 pts   (range -7.3 to +0.0)
  clips where they agree within 1 pt: 5/10

--- floor  (n=10) ---
  judge  mean WER  113.0 %
  small.en mean WER  101.0 %
  mean delta        +12.0 pts   (range -22.6 to +76.0)
  clips where they agree within 1 pt: 2/10



## 3. The divergence, in the error split

The total can hide the mechanism. Compare how each listener *fails*:

- **deletions** -- the listener gave up and omitted words
- **insertions** -- the listener produced words that are not in the reference,
  which on a two-speaker mixture usually means it transcribed the interferer too

A listener that omits and a listener that interleaves are doing different things
even when their totals match.

In [13]:
floor = frame[frame["system"] == "floor"]
if len(floor):
    print("On the unprocessed mixture (floor):")
    print(f"  small.en : {floor['asr_S'].sum():4d} sub  {floor['asr_D'].sum():4d} del  {floor['asr_I'].sum():4d} ins")
    print(f"  judge    : {floor['judge_S'].sum():4d} sub  {floor['judge_D'].sum():4d} del  {floor['judge_I'].sum():4d} ins")
    print()
    ratio_asr = floor["asr_I"].sum() / max(floor["asr_D"].sum(), 1)
    ratio_j   = floor["judge_I"].sum() / max(floor["judge_D"].sum(), 1)
    print(f"  insertions per deletion -- small.en {ratio_asr:.2f}, judge {ratio_j:.2f}")
    print("  A high ratio means the listener adds rather than omits, i.e. it")
    print("  transcribes both talkers instead of dropping the confusing parts.")

ceiling = frame[frame["system"] == "ceiling"]
if len(ceiling):
    print(f"\nOn the clean target (ceiling), n={len(ceiling)}:")
    print(f"  judge    mean {ceiling['judge_wer'].mean():.1f} %   max {ceiling['judge_wer'].max():.1f} %")
    print(f"  small.en mean {ceiling['asr_wer'].mean():.1f} %   max {ceiling['asr_wer'].max():.1f} %")
    print("\n  This is the CEILING: the best any system could score through this")
    print("  listener. Never quote a score without it. A lower ceiling means a")
    print("  wider measurable range between doing nothing and doing it perfectly.")

On the unprocessed mixture (floor):
  small.en :  149 sub    53 del    76 ins
  judge    :  111 sub    32 del   181 ins

  insertions per deletion -- small.en 1.43, judge 5.66
  A high ratio means the listener adds rather than omits, i.e. it
  transcribes both talkers instead of dropping the confusing parts.

On the clean target (ceiling), n=10:
  judge    mean 1.6 %   max 9.5 %
  small.en mean 3.8 %   max 9.5 %

  This is the CEILING: the best any system could score through this
  listener. Never quote a score without it. A lower ceiling means a
  wider measurable range between doing nothing and doing it perfectly.


## 4. Silence: does the judge invent words?

On an absent trial the clean target is digital silence, measured RMS 0. The only
correct answer is no words. This is the one place the offline ASR is known to
misbehave -- `small.en` emits the word "you" on silence in 8 of 8 absent trials
(`decisions-m3.md`), which is why that token has to be filtered before counting
invented words.

Per B4 these trials are reported on their own row and never folded into the
headline, because a rate of 0/0 is undefined rather than perfect.

In [14]:
silence = [r for r in records if r["system"] == "silence"]
if not silence:
    print("No silence clips judged yet. Run:")
    print("  python3 scripts/judge_smoke.py --n 10 --absent 2")
else:
    invented = 0
    for rec in silence:
        words = normalise_text(rec["judge"]).split()
        asr_words = normalise_text(rec["asr"] or "").split()
        bad = len(words) > 0
        invented += bad
        print(f"{rec['trial_id']}")
        print(f"  judge    : [{rec['judge_status']}] {len(words):2d} words  {words if words else '(silent -- correct)'}")
        print(f"  small.en : {len(asr_words):2d} words  {asr_words if asr_words else '(silent)'}")
        print(f"  -> judge {'INVENTED SPEECH' if bad else 'correctly stayed quiet'}")
        print()
    print(f"invented-speech rate, judge: {invented}/{len(silence)}"
          f" = {invented / len(silence) * 100:.0f} %")
    print("Lower is better. This is the tripwire for a judge that will not")
    print("report silence, which NRR cannot distinguish from a muting extractor.")

sir0_val-42-000000
  judge    : [speech] 42 words  ['and', 'that', 'is', 'why', 'they', 'say', 'success', 'does', 'not', 'require', 'excuse', 'what', 'you', 'just', 'need', 'to', 'do', 'is', 'to', 'be', 'consistent', 'with', 'yourself', 'and', 'keep', 'building', 'on', 'that', 'thing', 'you', 'are', 'building', 'on', 'in', 'no', 'time', 'you', 'see', 'yourself', 'at', 'the', 'top']
  small.en :  1 words  ['you']
  -> judge INVENTED SPEECH

sir0_val-42-000003
  judge    : [speech] 27 words  ['the', 'original', 'intention', 'of', 'keeping', 'them', 'alive', 'was', 'to', 'use', 'them', 'as', 'hostages', 'if', 'we', 'kill', 'them', 'now', 'would', 'not', 'that', 'mean', 'we', 'are', 'admitting', 'to', 'everything']
  small.en :  1 words  ['you']
  -> judge INVENTED SPEECH

invented-speech rate, judge: 2/2 = 100 %
Lower is better. This is the tripwire for a judge that will not
report silence, which NRR cannot distinguish from a muting extractor.


## 5. What to carry out of this

Fill these in from the cells above before quoting anything elsewhere:

- **Ceiling through the judge** -- and therefore the real measurable range.
- **Whether judge and ASR disagree**, and in which direction.
- **Whether the disagreement is in insertions or deletions**, which is the
  mechanism rather than the magnitude.
- **The invented-speech rate on silence.**

Every judge result must record the exact model ID, the exact prompt, the input
modality and the run date (CLAUDE.md). The first cell prints all four -- copy them
into `decisions-m4.md` alongside any number you quote, because closed models change
silently and a comparison across dates is invalid unless re-run.